# Week 2 Quiz — Advanced DataFrame API

**Topics:** UDFs, Window Functions, Aggregations, Joins, Collections, Caching

**12 questions.** Run each answer cell to reveal the correct answer and explanation.

---

## Q1

A data engineer has written a PySpark UDF that reads from a global list to look up values. A colleague warns this approach is problematic. Which of the following best describes the best practice for UDFs in Spark?

- A. UDFs can safely read and write to global mutable state shared across tasks
- B. UDFs should always return complex types such as structs or maps
- C. Pandas UDFs have higher per-row overhead than row-at-a-time UDFs
- **D. UDFs should be stateless and free of side effects to ensure correct and reproducible results**

In [ ]:
answer = "D"
explanation = (
    "UDFs run inside tasks distributed across multiple executors. Any mutable global state "
    "is NOT shared between tasks (each Python worker gets its own process), so writes are lost "
    "and reads can be stale. Stateless, side-effect-free UDFs guarantee deterministic results "
    "regardless of task retries or partition ordering."
)
print(f"Q1 → {answer}  |  {explanation}")

## Q2

A data engineer needs to count distinct values across a billion-row column. An exact count is not required — a close approximation is acceptable. Which function provides the best performance?

- A. `count(distinct col("id"))`
- B. `countDistinct(col("id"))`
- **C. `approx_count_distinct(col("id"))`**
- D. `size(collect_set(col("id")))`

In [ ]:
answer = "C"
explanation = (
    "approx_count_distinct() uses the HyperLogLog++ algorithm to estimate distinct counts with "
    "configurable relative error (default ~5%). It processes data in a single pass without "
    "shuffling all values to one partition, making it dramatically faster than exact "
    "countDistinct() or count(distinct ...) on large datasets. collect_set() would collect "
    "all values to the driver — extremely unsafe on large datasets."
)
print(f"Q2 → {answer}  |  {explanation}")

## Q3

A data engineer has a PySpark DataFrame with columns `product` and `revenue`. They need to compute each product's total revenue, average revenue, and transaction count. Which code snippet is correct?

- A. `df.select("product","revenue").groupby("product").agg(F.sum("revenue"), F.alias("total_revenue"), F.mean("revenue").alias("transaction_count"))`
- B. `df.groupby("product").agg(sum("revenue"), avg("revenue"), count("revenue"))`
- **C. `df.select("product","revenue").groupby("product").agg(F.sum("revenue"), F.mean("revenue"), F.count("revenue"))`**
- D. `df.groupby("product").agg(F.sum("revenue"), F.mean("revenue"), F.count("count"))`

In [ ]:
answer = "C"
explanation = (
    "Option C imports functions as F and correctly chains groupby('product').agg() with "
    "F.sum, F.mean, and F.count all applied to 'revenue'. "
    "Option A has a syntax error: F.alias() is called directly on the module, not on a column expression. "
    "Option B uses sum/avg/count without importing them. "
    "Option D calls F.count('count') referencing a non-existent column."
)
print(f"Q3 → {answer}  |  {explanation}")

## Q4

A global retail company has a DataFrame `sales_df` with columns `product_category`, `region`, and `sales_amount`. The team needs the total `sales_amount` per region stored in `region_sales`. Which code is correct?

- A. `region_sales = sales_df.groupby("region").agg(sum("sales_amount")).alias("total_sales_amount")`
- B. `region_sales = sales_df.groupby("region").agg(sum("sales_df.sales_amount")).alias("total_sales_amount")`
- C. `region_sales = sales_df.groupby("sales_amount"), groupby("category").alias("total_sales_amount")`
- **D. `region_sales = sales_df.groupby("region").agg(sum("sales_amount")).alias("total_sales_amount")`**

In [ ]:
answer = "D"
explanation = (
    "Option D correctly groups by 'region', aggregates with sum('sales_amount'), and assigns "
    "the result to 'region_sales' as required. (Ideally .alias() should go inside agg() on the "
    "column expression, but D is the only option with the correct variable name, groupby column, "
    "and aggregate.) Option B uses an invalid string format 'sales_df.sales_amount'. "
    "Option C chains a second comma-separated groupby, which is invalid syntax."
)
print(f"Q4 → {answer}  |  {explanation}")

## Q5

Which SQL keyword can be used to convert a table from **long format** (many rows, few columns) to **wide format** (fewer rows, many columns)?

- **A. `PIVOT`**
- B. `CONVERT`
- C. `WHERE`
- D. `TRANSFORM`
- E. `SUM`

In [ ]:
answer = "A"
explanation = (
    "The PIVOT clause in Spark SQL rotates rows into columns: distinct values in one column "
    "become new column headers, and an aggregate (e.g. SUM) fills each cell. This is the "
    "standard long-to-wide transformation. CONVERT changes data types; WHERE filters rows; "
    "TRANSFORM applies a function to array elements; SUM is an aggregate, not a reshaping keyword."
)
print(f"Q5 → {answer}  |  {explanation}")

## Q6

A data engineer runs the following query joining a `sales` table (LEFT) and `favorite_stores` table (RIGHT) on `customer_id`:

```sql
SELECT sale_id, customer_id, favorite_stores.stores_id
FROM sales
LEFT JOIN favorite_stores
ON sales.customer_id = favorite_stores.customer_id;
```

The `sales` table has rows for customer_ids 44, 23, 42. `favorite_stores` has a match for 44 (s1) and 42 (s2), but not 23. Which result is correct?

- A. Only the 2 matching rows (44 and 42) are returned
- B. All columns from both tables are returned for every row combination
- **C. All 3 sales rows are returned; the stores_id for customer 23 is NULL**
- D. Each sales row is repeated for every matching stores row

In [ ]:
answer = "C"
explanation = (
    "A LEFT JOIN returns ALL rows from the left table (sales) regardless of whether there "
    "is a match in the right table. For customer 23, which has no entry in favorite_stores, "
    "stores_id will be NULL. The result has 3 rows: (sale_id_44, 44, s1), (sale_id_23, 23, NULL), "
    "(sale_id_42, 42, s2). Option A describes an INNER JOIN. Option D describes a CROSS JOIN behavior."
)
print(f"Q6 → {answer}  |  {explanation}")

## Q7

A company has a `sales_df` with columns `product_category`, `region`, and `sales_amount`. The data engineer needs to analyze the sales amount for each category-region combination. Which code correctly groups by both columns and sums `sales_amount`?

- A. `category_sales = sales_df.groupby("category").agg(sum("sales_amount")).alias("total_sales_amount")`
- B. `category_sales = sales_df.sum("sales_amount"), groupby("category")`
- C. `category_sales = sales_df.groupby("sales_amount"), groupby("category").alias("total_sales_amount")`
- **D. `region_sales = sales_df.groupby("region").agg(sum("sales_amount")).alias("total_sales_amount")`**

In [ ]:
answer = "D"
explanation = (
    "Option D is the only syntactically correct groupBy + agg pattern and stores the result "
    "in 'region_sales'. Option A groups by 'category' instead of 'region'. "
    "Options B and C are invalid Python/PySpark syntax — they use commas to chain expressions "
    "in a non-callable way. The correct pattern is: "
    "df.groupBy('col').agg(sum('amount').alias('total'))."
)
print(f"Q7 → {answer}  |  {explanation}")

## Q8

Which window function assigns the same rank to tied values **but leaves gaps** in the ranking sequence after a tie?

- A. `row_number()`
- B. `dense_rank()`
- **C. `rank()`**
- D. `percent_rank()`

In [ ]:
answer = "C"
explanation = (
    "rank() assigns the same number to tied rows and then skips ranks equal to the number of "
    "tied rows. Example: two rows tied at rank 1 → both get 1, the next row gets 3 (gap). "
    "dense_rank() also ties equally but does NOT leave gaps (next rank is always +1). "
    "row_number() always assigns unique sequential numbers even for ties. "
    "percent_rank() computes a relative percentile (0.0 to 1.0)."
)
print(f"Q8 → {answer}  |  {explanation}")

## Q9

What is the default storage level used by the `cache()` method on a PySpark DataFrame?

- A. `MEMORY_ONLY`
- B. `DISK_ONLY`
- **C. `MEMORY_AND_DISK`**
- D. `OFF_HEAP`

In [ ]:
answer = "C"
explanation = (
    "df.cache() is a shortcut for df.persist(StorageLevel.MEMORY_AND_DISK). This means Spark "
    "first tries to store partitions in memory; if memory runs out, remaining partitions spill "
    "to disk. On recompute, Spark reads from disk rather than re-executing the lineage. "
    "MEMORY_ONLY would drop and recompute partitions that don't fit. "
    "Use persist(StorageLevel.MEMORY_ONLY) explicitly if you want pure in-memory caching."
)
print(f"Q9 → {answer}  |  {explanation}")

## Q10

What is the key difference between `collect_list()` and `collect_set()` aggregation functions in PySpark?

- A. `collect_list()` returns only distinct values; `collect_set()` preserves duplicates
- **B. `collect_list()` returns all values including duplicates; `collect_set()` returns only distinct values**
- C. Both functions return identical results regardless of duplicates
- D. `collect_list()` returns a sorted array; `collect_set()` returns an unsorted set

In [ ]:
answer = "B"
explanation = (
    "collect_list() gathers all values (preserving duplicates and order of encounter) into an "
    "array. collect_set() gathers only the unique values (deduplicates) into an unordered array. "
    "Neither function guarantees ordering. Both collect all values to a single executor so they "
    "should not be used on high-cardinality groups — they can cause OOM errors."
)
print(f"Q10 → {answer}  |  {explanation}")

## Q11

A DataFrame has a column `skills` containing arrays. Some rows have empty arrays `[]` and some have `null`. A data engineer wants to flatten the arrays into individual rows but **must keep rows with empty/null arrays** in the output (as a single row with `null`). Which function should they use?

- A. `explode(col("skills"))`
- B. `array_contains(col("skills"), "sql")`
- C. `flatten(col("skills"))`
- **D. `explode_outer(col("skills"))`**

In [ ]:
answer = "D"
explanation = (
    "explode() drops rows where the array is empty or null — those rows disappear entirely. "
    "explode_outer() preserves such rows, emitting a single row with null as the exploded value. "
    "This is the LEFT OUTER equivalent of explode and is critical when you need to audit "
    "records with missing array data. array_contains checks membership; flatten merges nested arrays."
)
print(f"Q11 → {answer}  |  {explanation}")

## Q12

Why do **pandas UDFs** (vectorized UDFs) generally outperform **row-at-a-time UDFs** in PySpark?

- A. Pandas UDFs run on the driver node instead of executors, avoiding network shuffles
- B. Pandas UDFs require no serialization at all
- **C. Pandas UDFs process data in columnar batches using Apache Arrow, drastically reducing Python–JVM serialization overhead**
- D. Pandas UDFs can safely read global mutable state on each executor

In [ ]:
answer = "C"
explanation = (
    "Row-at-a-time UDFs serialize each row from JVM to Python, invoke the Python function, "
    "then serialize the result back — paying the serialization cost per row. "
    "Pandas UDFs use Apache Arrow to transfer entire column batches in one zero-copy transfer, "
    "then the Python function operates on a pandas Series (vectorized NumPy operations). "
    "This reduces both per-row overhead and Python-JVM context switches, often giving 10-100x speedup."
)
print(f"Q12 → {answer}  |  {explanation}")